In [1]:
!python3 --version 

Python 3.10.11


In [2]:
!pip3 install transformers sentence_transformers faiss-cpu fastapi uvicorn pyngrok tf-keras

  Using cached transformers-4.56.2-py3-none-any.whl.metadata (40 kB)
  Using cached sentence_transformers-5.1.1-py3-none-any.whl.metadata (16 kB)
  Using cached faiss_cpu-1.12.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.1 kB)
  Using cached fastapi-0.118.0-py3-none-any.whl.metadata (28 kB)
  Using cached uvicorn-0.37.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached pyngrok-7.4.0-py3-none-any.whl.metadata (8.1 kB)
  Using cached tf_keras-2.20.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached huggingface_hub-0.35.3-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2025.9.18-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
  Using cached safetensors-0.6.2-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
 

In [4]:
# Cargar tokenizer y modelo de lenguaje localmente
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
#torch.set_num_threads(8)

tokenizer = AutoTokenizer.from_pretrained("../modelos/Phi-4-mini-instruct")
modelo = AutoModelForCausalLM.from_pretrained("../modelos/Phi-4-mini-instruct", dtype=torch.bfloat16).to("cuda")
#tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-4-mini-instruct")
#modelo = AutoModelForCausalLM.from_pretrained("microsoft/Phi-4-mini-instruct", dtype=torch.bfloat16).to("cuda")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
torch.get_num_threads()

16

In [5]:
from sentence_transformers import SentenceTransformer

# Cargar modelo localmente
embedder = SentenceTransformer("../modelos/all-MiniLM-L6-v2")

In [ ]:
import faiss
import numpy as np

embeddings = np.load("../embeddings/CHUNK.npy")

# Crear índice FAISS
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

In [7]:
import pandas as pd

df = pd.read_csv("../data/processed/productos_corpus.csv",delimiter=",")
df['CHUNK_DESCRIPCION'] = df['CHUNK_DESCRIPCION'].fillna('')
df['CHUNK_CARACTERISTICAS'] = df['CHUNK_CARACTERISTICAS'].fillna('')
df['CHUNK_OBSERVACIONES'] = df['CHUNK_OBSERVACIONES'].fillna('')
df['CHUNK_RECOMENDACIONES'] = df['CHUNK_RECOMENDACIONES'].fillna('')
tamanio = len(df)

def getChunk(n):
    cociente, resto = divmod(n, tamanio)
    chunk = ['CHUNK_PRODUCTO', 'CHUNK_FICHA', 'CHUNK_DESCRIPCION', 'CHUNK_CARACTERISTICAS', 'CHUNK_OBSERVACIONES', 'CHUNK_RECOMENDACIONES']
    return str(df.iloc[resto][chunk[cociente]])

In [1]:
import spacy
from spacy.lang.es.stop_words import STOP_WORDS

VERBOS_RUIDO = ["necesitar", "querer", "buscar", "requerir", "comprar", "utilizar",'cosito','adecuar','cualquier','funcionar','usar','']

nlp = spacy.load("es_core_news_sm")

def palabras_relevantes(texto):
    doc = nlp(texto)

    palabras = []
    for token in doc:
        if (token.pos_ in ["NOUN", "VERB", "PROPN"]) and token.text.lower() not in STOP_WORDS:
            lemma = token.lemma_.lower()
            if lemma not in VERBOS_RUIDO:
                palabras.append(lemma)

    return " ".join(palabras)

# Ejemplo
oracion = "dentro de mi casa tengo una ducha, necesito el cosito para la ducha que se rompio."
resultado = palabras_relevantes(oracion)
print(resultado)


casa ducha ducha rompio


In [ ]:
def modelx(pregunta, k=3, temperatura=0.2):
    #print('1.- encode')
    # Embed la pregunta

    pregunta = palabras_relevantes(pregunta)

    pregunta_emb = embedder.encode([pregunta], convert_to_numpy=True)

    # Buscar los k textos más cercanos
    distancias, indices = index.search(pregunta_emb, k)
    contexto = "\n".join([getChunk(i) for i in indices[0]])

    #print(contexto)
    #print(distancias)

    prompt = f"""
    CONTEXTO:
    {contexto}
    PREGUNTA: {pregunta}
    RESPUESTA:"""

   

    inputs = tokenizer(prompt, return_tensors="pt").to(modelo.device)
    #print('3.- generate')
    outputs = modelo.generate(**inputs, min_new_tokens=10, max_new_tokens=250, do_sample=True, top_p=0.5, temperature=temperatura)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [10]:
import re
def respuestaModel(pregunta,k=3,temperatura=0.2):
    tok = modelx(pregunta, k,temperatura)
    # Buscar la primera respuesta
    #match = re.search(r"RESPUESTA:\s*(.+?)(?=\n\s*PREGUNTA:|\Z)", tok, re.DOTALL | re.IGNORECASE)
    #return match.group(1).strip()

    respuestas = re.findall(r"RESPUESTA:\s*(.+?)(?=\n\s*PREGUNTA:|\n\s*CONTEXTO:|\Z)", tok, re.DOTALL | re.IGNORECASE)

    # Filtramos respuestas que al menos terminan en un número (como "179.9 soles")
    #respuestas_validas = [r.strip() for r in respuestas if re.search(r"\d+(\.\d+)?\s+soles", r)]
    return respuestas[0].strip()

In [ ]:
print(respuestaModel('Necesito un rodillo, uno bastante ancho y que las cerdas no se desgasten tan rapido.',3))

CONTEXTO
repuesto rodillo universal 9 de la marca tekno del area de pinturas para los accesorios para pintar de la linea rodillos y bandejas de procedencia importado. altura del producto 9.00 cm, ancho del producto 7.50 cm, profundidad del producto 22.90 cm, volumen del producto 1545.75, diametro del producto 0, ean 7799110003461, medidas 7.50x22.90x9.00, tipo de producto repuesto, modelo universal, color blanco, material polipropileno, garantía 1 año, procedencia importado, país de origen argentina. En Promart sabemos que te gusta darle tu toque personal a las cosas, por ello trae estos repuestos de rodillos. Pinta sin miedo a que se malogre. Lo mejor en herramientas para pintura solo lo encuentras en Promart.pe. Excelente calidad. Resistencia y durabilidad.. No aplica. Colocar de manera correcta.

RESPUESTA
Sí, claro, el rodillo universar 9 tiene las medidas de 7.50 de ancho, 22.90 de profundidad y 9 de alto. Es de polipropileno, duradero y viene en color blanco, además tenemos una g

In [ ]:
print(respuestaModel('Quiero una curva para cableado electrico',3))

CONTEXTO
curva sel 3/4" de la marca matusita del area de electricidad para los tuberias, canaletas y conexiones de la linea conexiones pvc de procedencia nacional. volumen del producto 0, ean 7755744011845, tipo de producto curva, modelo sel, color no aplica, material pvc, garantía 1 año, procedencia nacional, país de origen perú. . Liviano. Brinda menor pérdida de presión. Fácil instalación. Mayor vida útil. Resistencia a la corrosión interna y externa. Libre de olor, sabor o toxicidad. Químicamente inerte.. Utilizar el producto para la actividad que fue diseñado.. Verificar la medida del producto con aquellos que se desea unir. Mantener en buen estado, para garantizar un buen desempeño.

RESPUESTA
La curva sel 3/4" de Matusita es práctico para pasar cableado. Tiene medida 3/4". Es de pvc. Si ya tienes la base o el soporte, solo lo cambias y listo.


In [ ]:
print(respuestaModel('La bateria de mi carro no enciende el motor',3))

CONTEXTO
batería lms m95r i2 s09 de la marca enerjet del area de herramientas para los automotriz de la linea baterias de procedencia nacional. altura del producto 22.70 cm, ancho del producto 30.50 cm, profundidad del producto 17.00 cm, volumen del producto 11769.95, ean 7750424340161, medidas 30.50x17.00x22.70, tipo de producto batería, modelo m95rn, color negro, garantía 1 año, procedencia nacional. La batería es el elemento fundamental para el funcionamiento de un vehículo. Por ello, te traemos esta batería que reúne poder e innovación tecnológica en su interior, para asegurarte un óptimo desempeño.. Excelente calidad y funcionalidad a su alcance. Máxima potencia.. . Asegurarse de la potencia que necesita, antes de su instalación. Ideal para los usuarios de vehículos de uso particular o régimen de trabajo ligero.

RESPUESTA
La batería es el elemento fundamental para el funcionamiento de un vehículo, por ello, te traemos esta batería que reúne poder e innovación tecnológica en su in

In [ ]:
print(respuestaModel('Busco hornear un postre para un negocio, necesito un molde',3))

CONTEXTO
molde rectangular non stick gold 25cm de la marca ilko del area de cocina para los menaje de la linea vajilla y fuentes de procedencia nacional. altura del producto 7.00 cm, ancho del producto 12.00 cm, profundidad del producto 26.00 cm, volumen del producto 2184, diametro del producto 0, ean 7806810209250, medidas 12.00x26.00x7.00, tipo de producto molde, modelo non stick, color oro, material acero al carbono, garantía 2 años, procedencia nacional, país de origen chile. Renueva los utensilios de tu cocina con este molde rectangular de acero con recubrimiento antiadherente.. Espesor de 0.4 mm. Con recubrimiento antiadherente Withford.. Libre de PFOA.. Lavar antes del primer uso. Lavar cuidadosamente luego de cada uso. Lavar con detergente y esponja no abrasiva. Apto para lavavajillas pero se recomienda lavar a mano.

RESPUESTA
Para una mejor presentacion, este molde rectangular non stick gold marca Ilko viene en la dimensión 12.00x26.00x7.00. Si quieres, te armo el combo con u

In [ ]:
print(respuestaModel('Necesito cambiar la cortina de mi ducha',3))

CONTEXTO
cortina ducha peva 180x180 cm - flowers de la marca sm del area de baños para los organizacion de baño de la linea cortinas de baño de procedencia importado. altura del producto 180 cm, ancho del producto 180 cm, profundidad del producto 1 cm, volumen del producto 32400, ean 3664323176650, medidas 180x1x180, tipo de producto cortina, modelo peva, color multicolor, material eva, garantía por defecto de fabricación, procedencia importado, país de origen china. . . .

RESPUESTA
La cortina de ducha peva tiene un excelente diseño en flores y es ideal para baños interiores y extriores.
